# ETL Pipeline - Retail Analytics
Pipeline completo de ETL con visualización en tiempo real.

**Pasos:** Extract → Profile → Clean → Transform → Validate → Load → Queries

In [ ]:
# === SETUP ===
import os, sys, warnings
warnings.filterwarnings('ignore')

# Detect environment
IN_COLAB = 'google.colab' in sys.modules
BASE_DIR = os.getcwd()

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_DIR = '/content/drive/MyDrive/ETL-G1_2026-2_U1_Lab/1b'
    os.makedirs(f'{BASE_DIR}/data/output', exist_ok=True)
    os.makedirs(f'{BASE_DIR}/data/processed', exist_ok=True)
    os.makedirs(f'{BASE_DIR}/logs', exist_ok=True)

sys.path.insert(0, f'{BASE_DIR}/src')

import pandas as pd
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)

print(f'Base dir: {BASE_DIR}')
print('Environment ready')

In [ ]:
# === IMPORT MODULES ===
from extract import extract_all, extract_csv
from profile import profile_dataframe, save_profile_report
from clean import clean_transactions, clean_references
from transform import transform_all
from validate import validate_all
from load import load_to_csv, load_to_sqlite, load_references_to_sqlite, create_tables
from queries import run_queries
from pathlib import Path

DATA_DIR = Path(BASE_DIR) / 'data'
OUTPUT_DIR = DATA_DIR / 'output'
RAW_DIR = DATA_DIR / 'raw'

print('All modules imported successfully')

---
## STEP 1 — Initialize Database

In [ ]:
db_path = DATA_DIR / 'retail_analytics.db'
create_tables(db_path)
print(f'Database initialized: {db_path}')

---
## STEP 2 — Extract Data

In [ ]:
raw_data = extract_all(RAW_DIR)

# Transaction summary
csv_rows = len(extract_csv(RAW_DIR / 'sales_cali.csv'))
json_rows = len(raw_data['transactions']) - csv_rows
xml_rows = len(raw_data['transactions']) - csv_rows - json_rows + csv_rows

print(f'CSV  (Cali):     {csv_rows} rows')
print(f'JSON (Bogota):   {len(raw_data["transactions"])} total rows')
print(f'Total transactions: {len(raw_data["transactions"])} rows')
print(f'\nReference tables:')
print(f'  products:      {len(raw_data["products"])} rows')
print(f'  stores:        {len(raw_data["stores"])} rows')
print(f'  promotions:    {len(raw_data["promotions"])} rows')
print(f'  targets:       {len(raw_data["targets"])} rows')

print('\n--- Sample: Transactions (first 5 rows) ---')
raw_data['transactions'].head()

In [ ]:
print('--- Reference: Products ---')
display(raw_data['products'])
print('\n--- Reference: Stores ---')
display(raw_data['stores'])
print('\n--- Reference: Promotions ---')
display(raw_data['promotions'])
print('\n--- Reference: Monthly Targets ---')
display(raw_data['targets'])

---
## STEP 3 — Profile Data

In [ ]:
profiles = {}
for name, df in raw_data.items():
    profiles[name] = profile_dataframe(df, name)

# Save report
report_path = save_profile_report(profiles, OUTPUT_DIR)

# Print summary table
summary_rows = []
for name, prof in profiles.items():
    summary_rows.append({
        'Dataset': name,
        'Rows': prof['rows'],
        'Columns': prof['num_columns'],
        'Total Nulls': prof['total_nulls'],
        'Duplicates': prof['duplicates']
    })
display(pd.DataFrame(summary_rows))

print(f'\nFull report saved to: {report_path}')

In [ ]:
# Detailed profile: transactions
tp = profiles['transactions']
print('=== Transaction Profile Details ===')
print(f'Rows: {tp["rows"]}')
print(f'Duplicate sale_line_id: {tp.get("duplicate_sale_line_id", 0)}')
print(f'Invalid quantities: {tp.get("invalid_quantities", 0)} ({tp.get("invalid_quantity_pct", 0)}%)')
print(f'Invalid prices: {tp.get("invalid_prices", 0)} ({tp.get("invalid_price_pct", 0)}%)')
print(f'Invalid dates: {tp.get("invalid_dates", 0)} ({tp.get("invalid_date_pct", 0)}%)')

if tp['nulls_per_column']:
    print('\nNulls per column:')
    for col, count in tp['nulls_per_column'].items():
        print(f'  {col}: {count}')
else:
    print('\nNo null values found')

---
## STEP 4 — Clean Data

In [ ]:
n_before = len(raw_data['transactions'])

cleaned_transactions = clean_transactions(raw_data['transactions'])
cleaned_refs = clean_references(
    raw_data['products'], raw_data['stores'],
    raw_data['promotions'], raw_data['targets']
)

n_after = len(cleaned_transactions)
removed = n_before - n_after

print(f'Rows before cleaning: {n_before}')
print(f'Rows after cleaning:  {n_after}')
print(f'Rows removed:         {removed}')
print(f'  - Duplicates: {n_before - len(raw_data["transactions"].drop_duplicates(subset="sale_line_id"))}')
print(f'\nCleaned transactions sample:')
cleaned_transactions.head()

In [ ]:
# Verify case normalization
print('Store IDs:', sorted(cleaned_transactions['store_id'].unique()))
print('Product IDs:', sorted(cleaned_transactions['product_id'].unique()))
print('Payment methods:', sorted(cleaned_transactions['payment_method'].unique()))
print(f'\nDate range: {cleaned_transactions["sale_date"].min()} to {cleaned_transactions["sale_date"].max()}')

---
## STEP 5 — Transform Data

In [ ]:
integrated = transform_all(
    cleaned_transactions,
    cleaned_refs['products'],
    cleaned_refs['stores'],
    cleaned_refs['promotions'],
    cleaned_refs['targets']
)

print(f'Total rows after transform: {len(integrated)}')
print(f'Columns: {list(integrated.columns)}')
print(f'\nDerived columns sample:')
integrated[['sale_line_id', 'product_name', 'store_name', 'quantity',
            'unit_price', 'gross_sales', 'discount_amount', 'net_sales',
            'month', 'day_name']].head(10)

---
## STEP 6 — Validate Data

In [ ]:
validation = validate_all(integrated, cleaned_refs)

print(f'Validation passed: {validation["passed"]}')
if validation['errors']:
    for err in validation['errors']:
        print(f'  ERROR: {err}')
else:
    print('\nAll validations passed:')
    print('  - Unique sale_line_id')
    print('  - Foreign key integrity')
    print('  - Positive sales values')
    print('  - Formula correctness (net_sales = gross_sales - discount_amount)')

---
## STEP 7 — Load Data

In [ ]:
# CSV export
csv_path = DATA_DIR / 'processed' / 'sales_analytics.csv'
load_to_csv(integrated, csv_path)
print(f'CSV exported: {csv_path} ({len(integrated)} rows)')

# SQLite load
load_to_sqlite(integrated, 'sales_analytics', db_path)
print(f'Loaded into sales_analytics table')

# Reference tables
load_references_to_sqlite(cleaned_refs, db_path)
print(f'\nAll data loaded into {db_path}')

In [ ]:
# Verify database tables
import sqlite3
conn = sqlite3.connect(str(db_path))

tables = pd.read_sql_query("SELECT name FROM sqlite_master WHERE type='table'", conn)
print('Tables in database:')
display(tables)

for table in tables['name']:
    count = pd.read_sql_query(f'SELECT COUNT(*) as cnt FROM {table}', conn).iloc[0, 0]
    print(f'  {table}: {count} rows')

conn.close()

---
## STEP 8 — Analytical Queries

In [ ]:
results = run_queries(db_path)

for name, df in results.items():
    print(f'\n===== {name.upper().replace("_", " ")} =====')
    display(df)

In [ ]:
# Save all outputs
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
for f in OUTPUT_DIR.glob('*.csv'):
    f.unlink()
for name, df in results.items():
    df.to_csv(OUTPUT_DIR / f'{name}.csv', index=False)
print(f'Saved {len(results)} query result files to {OUTPUT_DIR}')

---
## STEP 9 — Charts & Visualizations

In [ ]:
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.figsize'] = (12, 5)

# 1. Top 10 products by net sales
top_products = results['top_products']
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

axes[0].barh(top_products['product_name'], top_products['total_net_sales'], color='steelblue')
axes[0].set_xlabel('Net Sales')
axes[0].set_title('Top 10 Products by Net Sales')
axes[0].invert_yaxis()

# 2. Sales by category
cat = results['sales_by_category']
axes[1].pie(cat['total_net_sales'], labels=cat['category'], autopct='%1.1f%%', startangle=90)
axes[1].set_title('Sales by Category')

plt.tight_layout()
plt.show()

In [ ]:
# 3. Sales by region
region = results['sales_by_region']
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(region['region'], region['total_net_sales'], color=['#2ecc71', '#e74c3c', '#3498db'])
axes[0].set_ylabel('Net Sales')
axes[0].set_title('Total Sales by Region')

# 4. Goal compliance
gc = results['goal_compliance'].copy()
gc['label'] = gc['store_name'] + ' (' + gc['month'] + ')'
colors = ['#2ecc71' if s == 'Met' else '#e74c3c' for s in gc['status']]
axes[1].barh(gc['label'], gc['achievement_pct'], color=colors)
axes[1].axvline(x=100, color='black', linestyle='--', label='100% target')
axes[1].set_xlabel('Achievement %')
axes[1].set_title('Goal Compliance by Store & Month')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# 5. Monthly performance
mp = results['monthly_performance']
fig, ax = plt.subplots(figsize=(12, 5))

x = range(len(mp))
width = 0.35
ax.bar([i - width/2 for i in x], mp['actual_sales'], width, label='Actual Sales', color='steelblue')
ax.bar([i + width/2 for i in x], mp['sales_target'], width, label='Target', color='lightcoral')
ax.set_xticks(x)
ax.set_xticklabels(mp['store_name'] + '\n' + mp['month'], rotation=45, ha='right')
ax.set_ylabel('Sales')
ax.set_title('Monthly Performance vs Targets')
ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
# 6. Regional analysis
ra = results['regional_analysis']
fig, ax = plt.subplots(figsize=(12, 5))

ax.bar(ra['store_name'], ra['total_net_sales'], color='teal')
ax.set_ylabel('Net Sales')
ax.set_title('Sales by Store (Regional Analysis)')
ax.set_xticklabels(ra['store_name'], rotation=45, ha='right')

plt.tight_layout()
plt.show()

In [ ]:
# 7. Payment method distribution
payment_dist = integrated['payment_method'].value_counts()
fig, ax = plt.subplots(figsize=(8, 5))

ax.pie(payment_dist.values, labels=payment_dist.index, autopct='%1.1f%%',
       colors=['#3498db', '#2ecc71', '#e74c3c', '#f39c12', '#9b59b6'])
ax.set_title('Payment Method Distribution')

plt.tight_layout()
plt.show()

In [ ]:
# 8. Day of week distribution
day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
day_dist = integrated['day_name'].value_counts().reindex(day_order)
fig, ax = plt.subplots(figsize=(10, 5))

ax.bar(day_dist.index, day_dist.values, color='mediumpurple')
ax.set_ylabel('Number of Transactions')
ax.set_title('Transactions by Day of Week')
ax.set_xticklabels(day_dist.index, rotation=45)

plt.tight_layout()
plt.show()

---
## Pipeline Summary

In [ ]:
print('=' * 60)
print('         ETL PIPELINE - COMPLETED SUCCESSFULLY')
print('=' * 60)
print(f'\n  Raw rows:       {len(raw_data["transactions"])}')
print(f'  Cleaned rows:   {n_after}')
print(f'  Rows removed:   {removed}')
print(f'  Integrated:     {len(integrated)}')
print(f'  Validation:     {"PASSED" if validation["passed"] else "FAILED"}')
print(f'\n  Database:       {db_path}')
print(f'  CSV output:     {csv_path}')
print(f'  Query results:  {OUTPUT_DIR}')
print(f'  Profile report: {report_path}')
print('\n  Files saved:')
for f in sorted(OUTPUT_DIR.glob('*.csv')):
    print(f'    - {f.name}')
print('=' * 60)